# 作业2.ipynb：作业一 all-reduce 通信实验

本实验对应 distributed_communication_single_node，不是 attention 或 Transformer 计时。原 notebook 保留，新增独立脚本 `allreduce_assignment1_benchmark.py` 和这份实验记录。

实验设备为单节点 Intel Xeon E5-2697 v4（18核36线程、约62 GiB内存），1张 RTX 3090；PyTorch 2.9.1+cu128。Gloo 使用 CPU 张量和本机 loopback 接口，每个 rank 是独立进程，PyTorch 算子线程数固定为1。

容量定义：MiB=2²⁰字节，GiB=2³⁰字节。每个 rank 的 FP32 张量分别为1、10、100 MiB和1024 MiB（1 GiB）。原示例把1024²字节写成MB、且最大配置写成1000，这里明确采用二进制单位，并将最后一项设为完整1 GiB。

每组先验证整个张量的 SUM 结果，再预热5次，barrier对齐后连续执行20次 all_reduce。对每个 rank 的总耗时除以20，再取各rank中的最大均值作为集体通信耗时；初始化、数据构造、正确性验证、barrier和结果汇总均不计时。每组在独立进程中运行，超时限制280秒，结束销毁进程组。

计时阶段使用零张量，避免反复SUM导致数值溢出；张量大小、dtype和通信操作保持不变。本结果是20次调用的平均耗时，不是一次调用，也没有测量单次耗时的标准差。

## Gloo + CPU：平均耗时（ms/次）

| 每个进程的张量大小 | 2进程 | 4进程 | 6进程 |
|---|---:|---:|---:|
| 1 MiB | 1.635 | 2.814 | 3.976 |
| 10 MiB | 10.577 | 16.268 | 20.394 |
| 100 MiB | 83.764 | 151.387 | 201.479 |
| 1 GiB | 851.856 | 1413.673 | 1908.523 |

## Gloo + CPU：算法带宽（GB/s）

| 每个进程的张量大小 | 2进程 | 4进程 | 6进程 |
|---|---:|---:|---:|
| 1 MiB | 0.641 | 0.373 | 0.264 |
| 10 MiB | 0.991 | 0.645 | 0.514 |
| 100 MiB | 1.252 | 0.693 | 0.520 |
| 1 GiB | 1.260 | 0.760 | 0.563 |

这里的算法带宽仅为“每rank张量字节数 / 单次耗时”，GB按10⁹字节计算，不是物理链路带宽或理论峰值。

## NCCL + GPU：平均耗时（ms/次）

| 每个进程的张量大小 | 2进程 | 4进程 | 6进程 |
|---|---:|---:|---:|
| 1 MiB | 硬件不足 | 硬件不足 | 硬件不足 |
| 10 MiB | 硬件不足 | 硬件不足 | 硬件不足 |
| 100 MiB | 硬件不足 | 硬件不足 | 硬件不足 |
| 1 GiB | 硬件不足 | 硬件不足 | 硬件不足 |

机器只有1张GPU，多进程NCCL需要各进程独占使用的GPU，不能把多个rank挤在同一张卡上作为上述多卡实验。

## 可用于报告的三句话分析

大张量配置的单次 all-reduce 耗时总体更长，因为需要传输和归约的数据更多，小张量则更容易受到调用、同步和进程调度开销影响。增加进程数不会自动加快 all-reduce：每个rank仍持有同样大的张量，同时更多进程竞争单机CPU、内存带宽和loopback通信资源，具体趋势应以实测表格为准。当前只有一张GPU，无法完成要求的2/4/6卡NCCL对比，因此不能从CPU数据判断两种后端的性能差距。

## 如何重新运行

```bash
cd ~/projects/diy-llm/coursework/assignment2-systems/cs336_systems
../.venv/bin/python -u allreduce_assignment1_benchmark.py
```

以后换到至少6卡的机器，脚本会自动运行完整NCCL组合；目前自动将其标为UNAVAILABLE。单组CPU测试示例：

```bash
../.venv/bin/python allreduce_assignment1_benchmark.py --case --backend gloo --world-size 4 --size-mib 100
```

方法参考：[PyTorch distributed官方文档](https://docs.pytorch.org/docs/stable/distributed.html)。


![实测图](allreduce_assignment1_results/allreduce_comparison.png)

In [ ]:
from pathlib import Path
import json, sys, subprocess
RERUN = False  # 改为True才会重新运行全部组合并覆盖结果
if RERUN:
    subprocess.run([sys.executable, "-u", "allreduce_assignment1_benchmark.py"], check=True)

data = [json.loads(p.read_text()) for p in Path("allreduce_assignment1_results").glob("*_p*MiB.json")]
for r in sorted(data, key=lambda r: (r["backend"], r["size_MiB"], r["world_size"])):
    print(r["backend"], r["world_size"], r["size_MiB"], r["status"],
          f"{r['mean_ms']:.3f} ms" if r["status"] == "OK" else r.get("reason", ""))
